<a href="https://colab.research.google.com/github/fathinahnj/unet-plankton-segmentation/blob/main/kfold_as_default.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ini coba2 saja 5-fold cross validation in an entirely new file. kita coba pake model baseline dulu. kalo ini berhasil, kita coba juga model yg augmented. kalo misal oke, jadikan ini sebagai default instead of the old files.

kalo nda berhasil, kita kembali ke old files, kita selipkan disitu kodenya

# Imports

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="Vjof9rBzJ2QeB82ApBRi")
project = rf.workspace("planktonclassification").project("instance-t20")
version = project.version(4)
dataset = version.download("coco-segmentation")

loading Roboflow workspace...
loading Roboflow project...


In [ ]:
%%capture
import os
import cv2
import numpy as np
import tensorflow as tf
import pandas as pd

from pycocotools.coco import COCO
from sklearn.model_selection import KFold

from tensorflow.keras.optimizers import Adam
from tensorflow.keras import layers, Model

from tensorflow.keras.applications import (
    ResNet50,
    EfficientNetB4,
    MobileNetV2
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


# Path & Config

In [ ]:
BASE_PATH = "/content/drive/MyDrive/PLANKTON/data/instance/test"
DATA_PATH = "/content/instance-t20-4/valid"

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 8
EPOCHS = 20
KFOLD = 5
LR = 1e-5

MODEL_SAVE_DIR = os.path.join(BASE_PATH, "kfold_models")

os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

In [ ]:
ENCODERS = {
    "resnet50": ResNet50,
    "efficientnetb4": EfficientNetB4,
    "mobilenetv2": MobileNetV2
}

# Load COCO Dataset

In [ ]:
coco_json_path = os.path.join(DATA_PATH, "_annotations.coco.json")

coco = COCO(coco_json_path)

all_images = coco.dataset["images"]
all_annotations = coco.dataset["annotations"]
categories = coco.dataset["categories"]

image_ids = np.array([img["id"] for img in all_images])

loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


In [ ]:
# shared coco object

coco_kfold_obj = COCO()
coco_kfold_obj.dataset = {
    "images":      all_images,
    "annotations": all_annotations,
    "categories":  categories
}
coco_kfold_obj.createIndex()

creating index...
index created!


# Image Path

In [ ]:
def get_image_path(file_name):
    return os.path.join(DATA_PATH, file_name)

# Mask Creation

In [ ]:
def create_mask(image_info, coco_instance):

    height = image_info["height"]
    width = image_info["width"]

    mask = np.zeros((height, width), dtype=np.uint8)

    ann_ids = coco_instance.getAnnIds(imgIds=image_info["id"])
    anns = coco_instance.loadAnns(ann_ids)

    for ann in anns:
        m = coco_instance.annToMask(ann)
        mask[m == 1] = 1

    return mask

In [ ]:
def load_image_and_mask(img_info, augment=False):
    """Load a single image + binary mask, optionally apply color augmentation."""
    img_path = os.path.join(DATA_PATH, img_info["file_name"])

    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE)) / 255.0

    mask = create_mask(img_info, coco_kfold_obj)
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    mask = np.expand_dims(mask, axis=-1)

    if augment:
        image = tf.image.random_brightness(image, max_delta=0.2)
        image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
        image = tf.image.random_saturation(image, lower=0.8, upper=1.2)
        image = tf.clip_by_value(image, 0.0, 1.0).numpy()

    return image, mask

In [ ]:
def load_image_and_mask_augmented(img_info, apply_augmentation=True):

    img_path = os.path.join(data_path, img_info["file_name"])

    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE)) / 255.0

    mask = create_mask(img_info, coco_kfold_obj)
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE),
                      interpolation=cv2.INTER_NEAREST)

    mask = np.expand_dims(mask, axis=-1)

    if apply_augmentation:
        image = augment_color(image)

    return image, mask

# tf.data Pipeline

In [ ]:
def build_dataset(img_subset, augment=False, batch_size=BATCH_SIZE):
    """
    Build a tf.data.Dataset from a list of COCO image dicts.
    Loads lazily — no full numpy array materialised in memory.
    """
    def generator():
        for img_info in img_subset:
            image, mask = load_image_and_mask(img_info, augment=augment)
            yield image.astype(np.float32), mask.astype(np.float32)

    dataset = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32),
            tf.TensorSpec(shape=(IMG_SIZE, IMG_SIZE, 1), dtype=tf.float32)
        )
    )

    dataset = (
        dataset
        .shuffle(buffer_size=len(img_subset), reshuffle_each_iteration=True)
        .batch(batch_size)
        .prefetch(tf.data.AUTOTUNE)
    )

    return dataset

# Dice Score

In [ ]:
def dice_score(y_true, y_pred):

    intersection = np.sum(y_true * y_pred)
    union = np.sum(y_true) + np.sum(y_pred)

    return (2. * intersection + 1e-7) / (union + 1e-7)

# Loss Function

In [ ]:
def bce_dice_loss(y_true, y_pred):

    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)

    intersection = tf.reduce_sum(y_true * y_pred)
    union = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred)

    dice = (2. * intersection + 1e-7) / (union + 1e-7)

    return bce + (1 - dice)

# UNet Builder + Encoders

In [ ]:
def build_unet_with_encoder(encoder_name):

    EncoderClass = ENCODERS[encoder_name]

    encoder = EncoderClass(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights="imagenet"
    )

    skips = []

    if encoder_name == "resnet50":
        skips = [
            encoder.get_layer("conv1_relu").output,
            encoder.get_layer("conv2_block3_out").output,
            encoder.get_layer("conv3_block4_out").output,
            encoder.get_layer("conv4_block6_out").output
        ]
        bridge = encoder.get_layer("conv5_block3_out").output

    elif encoder_name == "efficientnetb4":
        skips = [
            encoder.get_layer("block2a_activation").output,
            encoder.get_layer("block3a_activation").output,
            encoder.get_layer("block4a_activation").output,
            encoder.get_layer("block6a_activation").output
        ]
        bridge = encoder.get_layer("top_activation").output

    elif encoder_name == "mobilenetv2":
        skips = [
            encoder.get_layer("block_1_expand_relu").output,
            encoder.get_layer("block_3_expand_relu").output,
            encoder.get_layer("block_6_expand_relu").output,
            encoder.get_layer("block_13_expand_relu").output
        ]
        bridge = encoder.get_layer("block_16_project").output

    x = bridge

    for skip in reversed(skips):

        x = layers.UpSampling2D(size=(2, 2))(x)
        x = layers.Concatenate()([x, skip])
        x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)
        x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)

    outputs = layers.Conv2D(1, 1, activation="sigmoid")(x)

    model = Model(encoder.input, outputs)

    return model

# K-Fold Setup

In [ ]:
kf = KFold(
    n_splits=KFOLD,
    shuffle=True,
    random_state=42
)

encoder_results = {}

# Train & Val

## resnet50

In [ ]:
from tensorflow.keras.applications import ResNet50

def build_unet_resnet50():

    encoder = ResNet50(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights="imagenet"
    )

    skips = [
        encoder.get_layer("conv1_relu").output,
        encoder.get_layer("conv2_block3_out").output,
        encoder.get_layer("conv3_block4_out").output,
        encoder.get_layer("conv4_block6_out").output
    ]

    x = encoder.get_layer("conv5_block3_out").output

    for skip in reversed(skips):
        x = layers.UpSampling2D()(x)
        x = layers.Concatenate()([x, skip])
        x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)
        x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)



    # Add an additional upsampling block to match IMG_SIZE (224x224)
    x = layers.UpSampling2D(size=(2, 2))(x)
    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)

    outputs = layers.Conv2D(1, 1, activation="sigmoid")(x)

    return Model(encoder.input, outputs)

In [ ]:
resnet_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(image_ids)):

    print(f"\n[ResNet50] Fold {fold+1}")

    model = build_unet_resnet50()
    model.compile(optimizer=Adam(LR), loss=bce_dice_loss)

    train_ids = image_ids[train_idx]
    val_ids = image_ids[val_idx]

    X_train, Y_train = [], []

    for img in all_images:
        if img["id"] in train_ids:
            image, mask = load_image_and_mask(img)
            X_train.append(image)
            Y_train.append(mask)

    X_train = np.array(X_train)
    Y_train = np.array(Y_train)

    model.fit(X_train, Y_train, epochs=EPOCHS, batch_size=BATCH_SIZE)

    fold_dice = []

    for img in all_images:
        if img["id"] in val_ids:
            image, mask_true = load_image_and_mask(img)
            pred = model.predict(np.expand_dims(image, 0))[0,:,:,0]
            pred = (pred > 0.3).astype(np.uint8)

            d = dice_score(mask_true[:,:,0], pred)
            fold_dice.append(d)

    score = np.mean(fold_dice)
    resnet_scores.append(score)

    model.save(os.path.join(MODEL_SAVE_DIR, f"resnet_fold_{fold+1}.h5"))

print("ResNet Avg:", np.mean(resnet_scores))


[ResNet50] Fold 1
Epoch 1/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 183s 4s/step - loss: 1.2922
Epoch 2/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 462ms/step - loss: 1.0009
Epoch 3/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 466ms/step - loss: 0.5039
Epoch 4/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 479ms/step - loss: 0.3755
Epoch 5/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 493ms/step - loss: 0.3063
Epoch 6/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 493ms/step - loss: 0.2830
Epoch 7/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 481ms/step - loss: 0.2823
Epoch 8/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 469ms/step - loss: 0.2634
Epoch 9/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 467ms/step - loss: 0.2183
Epoch 10/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 463ms/step - loss: 0.2205
Epoch 11/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 465ms/step - loss: 0.2127
Epoch 12/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 466ms/step - loss: 0.2072
Epoch 13/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 467ms/step - loss: 0.1983
Epoch 14/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 473ms/step - loss: 0.1853
Epoch 15/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 


[ResNet50] Fold 2
Epoch 1/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 100s 2s/step - loss: 1.3114
Epoch 2/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 480ms/step - loss: 1.0809
Epoch 3/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 498ms/step - loss: 0.6356
Epoch 4/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 501ms/step - loss: 0.4303
Epoch 5/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 478ms/step - loss: 0.3314
Epoch 6/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 469ms/step - loss: 0.2889
Epoch 7/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 461ms/step - loss: 0.2648
Epoch 8/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 463ms/step - loss: 0.2572
Epoch 9/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 462ms/step - loss: 0.2253
Epoch 10/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 464ms/step - loss: 0.2235
Epoch 11/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 469ms/step - loss: 0.2304
Epoch 12/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 471ms/step - loss: 0.2112
Epoch 13/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 477ms/step - loss: 0.1896
Epoch 14/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 477ms/step - loss: 0.1799
Epoch 15/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 


[ResNet50] Fold 3
Epoch 1/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 99s 2s/step - loss: 1.4540
Epoch 2/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 478ms/step - loss: 1.1101
Epoch 3/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 490ms/step - loss: 0.7529
Epoch 4/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 502ms/step - loss: 0.4526
Epoch 5/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 484ms/step - loss: 0.3699
Epoch 6/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 470ms/step - loss: 0.3502
Epoch 7/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 465ms/step - loss: 0.2988
Epoch 8/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 461ms/step - loss: 0.2971
Epoch 9/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 462ms/step - loss: 0.2482
Epoch 10/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 463ms/step - loss: 0.2481
Epoch 11/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 468ms/step - loss: 0.2270
Epoch 12/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 475ms/step - loss: 0.2214
Epoch 13/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 473ms/step - loss: 0.2151
Epoch 14/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 479ms/step - loss: 0.1910
Epoch 15/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8


[ResNet50] Fold 4
Epoch 1/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 135s 4s/step - loss: 1.2109
Epoch 2/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 463ms/step - loss: 0.8333
Epoch 3/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 470ms/step - loss: 0.4771
Epoch 4/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 477ms/step - loss: 0.3401
Epoch 5/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 494ms/step - loss: 0.3397
Epoch 6/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 502ms/step - loss: 0.3246
Epoch 7/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 491ms/step - loss: 0.3570
Epoch 8/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 483ms/step - loss: 0.2743
Epoch 9/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 471ms/step - loss: 0.2370
Epoch 10/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 471ms/step - loss: 0.2393
Epoch 11/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 467ms/step - loss: 0.2143
Epoch 12/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 469ms/step - loss: 0.2002
Epoch 13/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 473ms/step - loss: 0.1946
Epoch 14/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 474ms/step - loss: 0.1828
Epoch 15/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 


[ResNet50] Fold 5
Epoch 1/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 102s 2s/step - loss: 1.3321
Epoch 2/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 482ms/step - loss: 1.0032
Epoch 3/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 496ms/step - loss: 0.5564
Epoch 4/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 508ms/step - loss: 0.3967
Epoch 5/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 491ms/step - loss: 0.3085
Epoch 6/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 476ms/step - loss: 0.3094
Epoch 7/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 472ms/step - loss: 0.2543
Epoch 8/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 466ms/step - loss: 0.2348
Epoch 9/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 466ms/step - loss: 0.2392
Epoch 10/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 470ms/step - loss: 0.2488
Epoch 11/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 467ms/step - loss: 0.2791
Epoch 12/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 479ms/step - loss: 0.2220
Epoch 13/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 480ms/step - loss: 0.2101
Epoch 14/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 483ms/step - loss: 0.1961
Epoch 15/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 

ResNet Avg: 0.000555076921822799


## efficientnetb4

In [ ]:
from tensorflow.keras.applications import EfficientNetB4

def build_unet_efficientnet():

    encoder = EfficientNetB4(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights="imagenet"
    )

    skips = [
        encoder.get_layer("block2a_activation").output,
        encoder.get_layer("block3a_activation").output,
        encoder.get_layer("block4a_activation").output
        # Removed block6a_activation as it causes a spatial mismatch
    ]

    x = encoder.get_layer("top_activation").output

    for skip in reversed(skips):
        x = layers.UpSampling2D(size=(2, 2))(x)
        x = layers.Concatenate()([x, skip])
        x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)
        x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)

    # After the loop, x is 56x56. Need to upsample to 224x224.
    # First upsample: 56x56 -> 112x112
    x = layers.UpSampling2D(size=(2, 2))(x)
    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)

    # Second upsample: 112x112 -> 224x224
    x = layers.UpSampling2D(size=(2, 2))(x)
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.Conv2D(16, 3, padding="same", activation="relu")(x)

    outputs = layers.Conv2D(1, 1, activation="sigmoid")(x)

    return Model(encoder.input, outputs)

In [ ]:
efficientnet_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(image_ids)):

    print(f"\n[EfficientNetB4] Fold {fold+1}")

    model = build_unet_efficientnet()
    model.compile(optimizer=Adam(LR), loss=bce_dice_loss)

    train_ids = image_ids[train_idx]
    val_ids = image_ids[val_idx]

    X_train, Y_train = [], []

    for img in all_images:
        if img["id"] in train_ids:
            image, mask = load_image_and_mask(img)
            X_train.append(image)
            Y_train.append(mask)

    X_train = np.array(X_train)
    Y_train = np.array(Y_train)

    model.fit(X_train, Y_train, epochs=EPOCHS, batch_size=BATCH_SIZE)

    fold_dice = []

    for img in all_images:
        if img["id"] in val_ids:
            image, mask_true = load_image_and_mask(img)
            pred = model.predict(np.expand_dims(image, 0))[0,:,:,0]
            pred = (pred > 0.3).astype(np.uint8)

            d = dice_score(mask_true[:,:,0], pred)
            fold_dice.append(d)

    score = np.mean(fold_dice)
    efficientnet_scores.append(score)

    model.save(os.path.join(MODEL_SAVE_DIR, f"efficientnet_fold_{fold+1}.h5"))

print("EfficientNet Avg:", np.mean(efficientnet_scores))


[EfficientNetB4] Fold 1
Epoch 1/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 302s 7s/step - loss: 1.2357
Epoch 2/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 248ms/step - loss: 0.8632
Epoch 3/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 249ms/step - loss: 0.5823
Epoch 4/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 255ms/step - loss: 0.4703
Epoch 5/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 254ms/step - loss: 0.4199
Epoch 6/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 253ms/step - loss: 0.3682
Epoch 7/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 263ms/step - loss: 0.3877
Epoch 8/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 258ms/step - loss: 0.3560
Epoch 9/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 252ms/step - loss: 0.3339
Epoch 10/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 252ms/step - loss: 0.3171
Epoch 11/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 254ms/step - loss: 0.3310
Epoch 12/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 246ms/step - loss: 0.2953
Epoch 13/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 245ms/step - loss: 0.3103
Epoch 14/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 257ms/step - loss: 0.3176
Epoch 15/20
17/17 ━━━━━━━━━━━━━━━


[EfficientNetB4] Fold 2
Epoch 1/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 236s 5s/step - loss: 1.2901
Epoch 2/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 51s 255ms/step - loss: 0.8549
Epoch 3/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 252ms/step - loss: 0.6177
Epoch 4/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 252ms/step - loss: 0.5206
Epoch 5/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 5s 266ms/step - loss: 0.4977
Epoch 6/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 259ms/step - loss: 0.4116
Epoch 7/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 261ms/step - loss: 0.4026
Epoch 8/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 5s 267ms/step - loss: 0.3722
Epoch 9/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 260ms/step - loss: 0.3424
Epoch 10/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 250ms/step - loss: 0.3162
Epoch 11/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 256ms/step - loss: 0.3105
Epoch 12/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 5s 247ms/step - loss: 0.2942
Epoch 13/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 246ms/step - loss: 0.2980
Epoch 14/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 251ms/step - loss: 0.3159
Epoch 15/20
17/17 ━━━━━━━━━━━━━━


[EfficientNetB4] Fold 3
Epoch 1/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 237s 5s/step - loss: 1.4048
Epoch 2/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 5s 270ms/step - loss: 1.0557
Epoch 3/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 5s 309ms/step - loss: 0.7477
Epoch 4/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 6s 357ms/step - loss: 0.5622
Epoch 5/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 254ms/step - loss: 0.4816
Epoch 6/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 254ms/step - loss: 0.4316
Epoch 7/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 256ms/step - loss: 0.4160
Epoch 8/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 250ms/step - loss: 0.3690
Epoch 9/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 247ms/step - loss: 0.3762
Epoch 10/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 256ms/step - loss: 0.3411
Epoch 11/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 245ms/step - loss: 0.3283
Epoch 12/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 244ms/step - loss: 0.3009
Epoch 13/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 259ms/step - loss: 0.3052
Epoch 14/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 245ms/step - loss: 0.3015
Epoch 15/20
17/17 ━━━━━━━━━━━━━━━


[EfficientNetB4] Fold 4
Epoch 1/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 277s 7s/step - loss: 1.5348
Epoch 2/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 246ms/step - loss: 1.3107
Epoch 3/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 255ms/step - loss: 0.9645
Epoch 4/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 257ms/step - loss: 0.6338
Epoch 5/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 253ms/step - loss: 0.5295
Epoch 6/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 263ms/step - loss: 0.4607
Epoch 7/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 5s 266ms/step - loss: 0.4292
Epoch 8/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 262ms/step - loss: 0.3898
Epoch 9/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 260ms/step - loss: 0.3660
Epoch 10/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 5s 270ms/step - loss: 0.4001
Epoch 11/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 252ms/step - loss: 0.3851
Epoch 12/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 253ms/step - loss: 0.3309
Epoch 13/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 5s 266ms/step - loss: 0.3311
Epoch 14/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 250ms/step - loss: 0.3360
Epoch 15/20
17/17 ━━━━━━━━━━━━━━━


[EfficientNetB4] Fold 5
Epoch 1/20


## mobilenetv2

In [ ]:
from tensorflow.keras.applications import MobileNetV2

def build_unet_mobilenet():

    encoder = MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights="imagenet"
    )

    skips = [
        encoder.get_layer("block_1_expand_relu").output,
        encoder.get_layer("block_3_expand_relu").output,
        encoder.get_layer("block_6_expand_relu").output,
        encoder.get_layer("block_13_expand_relu").output
    ]

    x = encoder.get_layer("block_16_project").output

    for skip in reversed(skips):
        x = layers.UpSampling2D()(x)
        x = layers.Concatenate()([x, skip])
        x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)
        x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)

    outputs = layers.Conv2D(1, 1, activation="sigmoid")(x)

    return Model(encoder.input, outputs)

In [ ]:
mobilenet_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(image_ids)):

    print(f"\n[MobileNetV2] Fold {fold+1}")

    model = build_unet_mobilenet()
    model.compile(optimizer=Adam(LR), loss=bce_dice_loss)

    train_ids = image_ids[train_idx]
    val_ids = image_ids[val_idx]

    X_train, Y_train = [], []

    for img in all_images:
        if img["id"] in train_ids:
            image, mask = load_image_and_mask(img)
            X_train.append(image)
            Y_train.append(mask)

    X_train = np.array(X_train)
    Y_train = np.array(Y_train)

    model.fit(X_train, Y_train, epochs=EPOCHS, batch_size=BATCH_SIZE)

    fold_dice = []

    for img in all_images:
        if img["id"] in val_ids:
            image, mask_true = load_image_and_mask(img)
            pred = model.predict(np.expand_dims(image, 0))[0,:,:,0]
            pred = (pred > 0.3).astype(np.uint8)

            d = dice_score(mask_true[:,:,0], pred)
            fold_dice.append(d)

    score = np.mean(fold_dice)
    mobilenet_scores.append(score)

    model.save(os.path.join(MODEL_SAVE_DIR, f"mobilenet_fold_{fold+1}.h5"))

print("MobileNet Avg:", np.mean(mobilenet_scores))

## final comparison

In [ ]:
df = pd.DataFrame({
    "Encoder": ["ResNet50", "EfficientNetB4", "MobileNetV2"],
    "Average Dice": [
        np.mean(resnet_scores),
        np.mean(efficientnet_scores),
        np.mean(mobilenet_scores)
    ],
    "Std Dice": [
        np.std(resnet_scores),
        np.std(efficientnet_scores),
        np.std(mobilenet_scores)
    ]
})

df = df.sort_values("Average Dice", ascending=False)
df

## na gabung jadi 1

In [ ]:
for encoder_name in ENCODERS.keys():

    print(f"\n==============================")
    print(f"TRAINING ENCODER: {encoder_name}")
    print(f"==============================")

    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(image_ids)):

        print(f"\n----- Fold {fold+1} -----")

        train_ids = image_ids[train_idx]
        val_ids = image_ids[val_idx]

        model = build_unet_with_encoder(encoder_name)

        model.compile(
            optimizer=Adam(LR),
            loss=bce_dice_loss
        )

        train_images = []
        train_masks = []

        for img in all_images:

            if img["id"] in train_ids:
                image, mask = load_image_and_mask(img)
                train_images.append(image)
                train_masks.append(mask)

        train_images = np.array(train_images)
        train_masks = np.array(train_masks)

        model.fit(
            train_images,
            train_masks,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            verbose=1
        )

        fold_dice = []

        for img in all_images:

            if img["id"] in val_ids:

                image, mask_true = load_image_and_mask(img)

                image = np.expand_dims(image, axis=0)

                pred = model.predict(image, verbose=0)
                pred = (pred > 0.3).astype(np.uint8)[0,:,:,0]

                d = dice_score(mask_true[:,:,0], pred)
                fold_dice.append(d)

        mean_dice = np.mean(fold_dice)
        fold_scores.append(mean_dice)

        print("Fold Dice:", mean_dice)

        model.save(os.path.join(
            MODEL_SAVE_DIR,
            f"{encoder_name}_fold_{fold+1}.h5"
        ))

    encoder_results[encoder_name] = fold_scores


TRAINING ENCODER: resnet50

----- Fold 1 -----
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 184s 5s/step - loss: 1.2454
Epoch 2/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 496ms/step - loss: 0.9035
Epoch 3/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 505ms/step - loss: 0.4805
Epoch 4/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 522ms/step - loss: 0.4133
Epoch 5/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 515ms/step - loss: 0.3031
Epoch 6/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 504ms/step - loss: 0.2834
Epoch 7/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 500ms/step - loss: 0.2548
Epoch 8/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 491ms/step - loss: 0.2530
Epoch 9/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 494ms/step - loss: 0.2295
Epoch 10/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 495ms/step - loss: 0.2294
Epoch 11/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 500ms/step - loss: 0.2030
Epoch 12/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 505ms/step - loss: 0.1858
Epoch 13/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 500ms/step - loss: 0.1677
Epoch 14/20
17/17 ━━━━━━━━━

Fold Dice: 6.015463701462838e-11

----- Fold 2 -----
Epoch 1/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 107s 2s/step - loss: 1.1383
Epoch 2/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 495ms/step - loss: 0.7032
Epoch 3/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 505ms/step - loss: 0.3992
Epoch 4/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 11s 525ms/step - loss: 0.3588
Epoch 5/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 518ms/step - loss: 0.3008
Epoch 6/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 506ms/step - loss: 0.2613
Epoch 7/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 500ms/step - loss: 0.2682
Epoch 8/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 491ms/step - loss: 0.2307
Epoch 9/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 493ms/step - loss: 0.2403
Epoch 10/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 491ms/step - loss: 0.2352
Epoch 11/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 496ms/step - loss: 0.2223
Epoch 12/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 502ms/step - loss: 0.2010
Epoch 13/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 501ms/step - loss: 0.1996
Epoch 14/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 504ms/step - loss: 0.1806
Epoc

Fold Dice: 4.71517452621683e-11

----- Fold 3 -----
Epoch 1/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 100s 2s/step - loss: 1.3175
Epoch 2/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 28s 500ms/step - loss: 1.0965
Epoch 3/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 10s 515ms/step - loss: 0.6719
Epoch 4/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 525ms/step - loss: 0.4525
Epoch 5/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 515ms/step - loss: 0.3325
Epoch 6/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 502ms/step - loss: 0.2902
Epoch 7/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 497ms/step - loss: 0.2611
Epoch 8/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 488ms/step - loss: 0.2479
Epoch 9/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 492ms/step - loss: 0.2289
Epoch 10/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 493ms/step - loss: 0.2217
Epoch 11/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 498ms/step - loss: 0.2319
Epoch 12/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 502ms/step - loss: 0.2219
Epoch 13/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 502ms/step - loss: 0.1953
Epoch 14/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 509ms/step - loss: 0.1929
Epoc

Fold Dice: 4.802275441775151e-11

----- Fold 4 -----
Epoch 1/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 140s 5s/step - loss: 1.2596
Epoch 2/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 482ms/step - loss: 0.9795
Epoch 3/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 493ms/step - loss: 0.5302
Epoch 4/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 506ms/step - loss: 0.4031
Epoch 5/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 518ms/step - loss: 0.4171
Epoch 6/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 529ms/step - loss: 0.3311
Epoch 7/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 518ms/step - loss: 0.2925
Epoch 8/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 506ms/step - loss: 0.2556
Epoch 9/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 502ms/step - loss: 0.2742
Epoch 10/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 496ms/step - loss: 0.2324
Epoch 11/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 498ms/step - loss: 0.2248
Epoch 12/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 499ms/step - loss: 0.1964
Epoch 13/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 504ms/step - loss: 0.2032
Epoch 14/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 506ms/step - loss: 0.1860
Epoch

Fold Dice: 5.904451968690509e-11

----- Fold 5 -----
Epoch 1/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 105s 2s/step - loss: 1.2703
Epoch 2/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 504ms/step - loss: 0.9397
Epoch 3/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 517ms/step - loss: 0.4942
Epoch 4/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 528ms/step - loss: 0.4021
Epoch 5/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 524ms/step - loss: 0.4143
Epoch 6/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 505ms/step - loss: 0.3299
Epoch 7/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 504ms/step - loss: 0.2969
Epoch 8/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 499ms/step - loss: 0.2700
Epoch 9/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 497ms/step - loss: 0.2390
Epoch 10/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 501ms/step - loss: 0.2463
Epoch 11/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 505ms/step - loss: 0.2660
Epoch 12/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 507ms/step - loss: 0.2321
Epoch 13/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 515ms/step - loss: 0.2000
Epoch 14/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 508ms/step - loss: 0.1861
Epoch

Fold Dice: 5.7022476103822407e-11

TRAINING ENCODER: efficientnetb4

----- Fold 1 -----
71686520/71686520 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


ValueError: A `Concatenate` layer requires inputs with matching shapes except for the concatenation axis. Received: input_shape=[(None, 14, 14, 1792), (None, 7, 7, 960)]

# Result Comparison

In [ ]:
comparison = []

for encoder, scores in encoder_results.items():

    comparison.append({
        "Encoder": encoder,
        "Fold1": scores[0],
        "Fold2": scores[1],
        "Fold3": scores[2],
        "Fold4": scores[3],
        "Fold5": scores[4],
        "Average Dice": np.mean(scores),
        "Std Dice": np.std(scores)
    })

df = pd.DataFrame(comparison)
df = df.sort_values("Average Dice", ascending=False)

df